# Lecture 6 - (18/06/2026)

Today's Topics:

- Linear Seperability
- Classification
- Linear Algebra Review
- Support Vector Machines

## Linear Seperability

### Hyperplanes

A **decision rule** tells us how to interpret the output of the model to make a decision on how to classify a datapoint. We commonly make decision rules by specifying a **threshold**, $T$. If the predicted probability is greater than or equal to $T$, predict Class 1. Otherwise, predict Class 0.

$$ \hat y = \text{classify}(x) = \begin{cases} 
\text{Class 1}, & p \ge T \\
\text{Class 0}, & p < T
\end{cases}$$

Using our decision rule, we can define a decision boundary as the “line” that splits the data into classes based on its features. For logistic regression, since we are working in $p$ dimensions, the decision boundary is a hyperplane --  a linear combination of the features in $p$-dimensions -- and we can recover it from the final logistic regression model.

In [ ]:
T = 0.5
x = np.random.uniform(-5, 5, 50)
y = np.array([0 if xi < T else 1 for xi in x])

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x,
    y=np.zeros_like(x),
    mode='markers',
    marker=dict(color=y, colorscale='Viridis', size=12),
    name='Data points'
))

fig.add_shape(
    type="line",
    x0=T, x1=T,
    y0=-0.5, y1=0.5,
    line=dict(color="red", width=3, dash="dash"),
    name='Hyperplane'
)

fig.add_annotation(
    x=T, y=0.3, text="Hyperplane", showarrow=True, arrowhead=2, arrowcolor="red"
)

fig.update_layout(
    title="1D Hyperplane",
    xaxis_title="x",
    yaxis=dict(showticklabels=False),
    height=300,
    width=700
)

fig.show()

In [ ]:
n = 50

x1 = np.random.uniform(-5, 5, n)
x2 = np.random.uniform(-5, 5, n)

w = np.array([1, 1])
T = 0.5

y = np.array([0 if w[0]*xi + w[1]*zi < T else 1 for xi, zi in zip(x1, x2)])

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x1,
    y=x2,
    mode='markers',
    marker=dict(color=y, colorscale='Viridis', size=10),
    name='Data points'
))

x_vals = np.linspace(-5, 5, 100)
y_vals = (T - w[0]*x_vals)/w[1]

fig.add_trace(go.Scatter(
    x=x_vals,
    y=y_vals,
    mode='lines',
    line=dict(color='red', width=3, dash='dash'),
    name='Hyperplane'
))

fig.update_layout(
    title="2D Hyperplane",
    xaxis_title="x1",
    yaxis_title="x2",
    width=700,
    height=600
)

fig.show()

In [ ]:
n = 50
x1 = np.random.uniform(-5, 5, n)
x2 = np.random.uniform(-5, 5, n)
x3 = np.random.uniform(-5, 5, n)

w = np.array([1, 1, 1])
T = 0.5

y = np.array([0 if w[0]*xi + w[1]*zi + w[2]*zi2 < T else 1 
              for xi, zi, zi2 in zip(x1, x2, x3)])

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=x1,
    y=x2,
    z=x3,
    mode='markers',
    marker=dict(color=y, colorscale='Viridis', size=5),
    name='Data points'
))

xx, yy = np.meshgrid(np.linspace(-5, 5, 10), np.linspace(-5, 5, 10))
zz = (T - w[0]*xx - w[1]*yy) / w[2]

fig.add_trace(go.Surface(
    x=xx, y=yy, z=zz,
    opacity=0.5,
    colorscale=[[0, 'red'], [1, 'red']],
    showscale=False,
    name='Hyperplane'
))

fig.update_layout(
    title="3D Hyperplane Example",
    scene=dict(
        xaxis_title='x1',
        yaxis_title='x2',
        zaxis_title='x3'
    ),
    width=800,
    height=700
)

fig.show()

In real life, however, that is often not the case, and we often see some overlap between points of different classes across the decision boundary. The true classes of the 2D data are shown below:

In [ ]:

n = 50
x1 = np.random.uniform(-5, 5, n)
x2 = np.random.uniform(-5, 5, n)

w = np.array([1, 1])
T = 0.5

linear_comb = w[0]*x1 + w[1]*x2

noise = np.random.normal(0, 4.0, n)
x2_noisy = x2 + noise
y = np.array([0 if val < T else 1 for val in linear_comb])

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x1,
    y=x2_noisy,
    mode='markers',
    marker=dict(color=y, colorscale='Viridis', size=10),
    name='Data points'
))

x_vals = np.linspace(-5, 5, 100)
y_vals = (T - w[0]*x_vals)/w[1]

fig.add_trace(go.Scatter(
    x=x_vals,
    y=y_vals,
    mode='lines',
    line=dict(color='red', width=3, dash='dash'),
    name='Hyperplane'
))

fig.update_layout(
    title="2D Hyperplane with Noise",
    xaxis_title="x1",
    yaxis_title="x2",
    width=700,
    height=600
)

fig.show()

As you can see, the decision boundary predicted by our logistic regression does not perfectly separate the two classes. There’s a “muddled” region near the decision boundary where our classifier predicts the wrong class. What would the data have to look like for the classifier to make perfect predictions?

[Hyperplane Playground](https://playground.tensorflow.org/#activation=tanh&batchSize=10&dataset=gauss&regDataset=reg-plane&learningRate=0.03&regularizationRate=0&noise=0&networkShape=4,2&seed=0.32912&showTestData=false&discretize=false&percTrainData=50&x=true&y=true&xTimesY=false&xSquared=false&ySquared=false&cosX=false&sinX=false&cosY=false&sinY=false&collectStats=false&problem=classification&initZero=false&hideText=false)

A classification dataset is said to be linearly separable if there exists a hyperplane among input features $x$ that separates the two classes $y$.

This same definition holds in higher dimensions. If there are two features, the separating hyperplane must exist in two dimensions (any line of the form $y=mx+b$). We can visualize this using a scatter plot.

When the dataset is linearly separable, a logistic regression classifier can perfectly assign datapoints into classes.

Here's the question, can it achieve 0 cross entropy loss?

$$ -(y \log(p) + (1-y) \log(1-p) )$$

Cross entropy loss is 0 if p=1 when y=1, and p=0 when y=0. 

This should be great! We can have 0 loss when we train the data on perfectly seperable data, however unxpected complications may arise.

$$ \sigma(\theta x) = \frac{1}{1+e^{- \theta x}}$$

In [ ]:
x = np.linspace(-10, 10, 400)
y = 1 / (1 + np.exp(-x))

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x,
    y=y,
    mode='lines',
    line=dict(color='blue', width=3),
    name='Sigmoid'
))

fig.update_layout(
    title='Sigmoid Function',
    xaxis_title='x',
    yaxis_title='σ(x)',
    width=700,
    height=500
)

fig.show()

The sigmoid can never output exactly 0 or 1, so no finite optimal $\theta$ exists. When the data is linearly seperable, the optimal model parameters diverge to $\pm \infty$.

In order to deal with this situation, we employ the use of $L_1$ and $L_2$ regularization to prevent overfitting.

## Classification

In a classification problem; we predict a category, not a continuous number as we do in regression.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression

url = "https://raw.githubusercontent.com/KeithGalli/pandas/master/pokemon_data.csv"
df = pd.read_csv(url)

df["Total"] = df[["HP","Attack","Defense","Sp. Atk","Sp. Def","Speed"]].sum(axis=1)

y = df["Legendary"].astype(int)
X = df[["Total"]]

lin = LinearRegression().fit(X, y)

x_range = np.linspace(X["Total"].min(), X["Total"].max(), 300)
x_range_df = pd.DataFrame({"Total": x_range})

lin_pred = lin.predict(x_range_df)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df["Total"],
    y=y,
    mode="markers",
    text=df["Name"],
    name="Pokemon",
    marker=dict(size=8)
))

fig.add_trace(go.Scatter(
    x=x_range.flatten(),
    y=lin_pred,
    mode="lines",
    name="Linear Regression",
    line=dict(width=4)
))

fig.update_layout(
    title="Predicting Legendary Pokemon using Regression",
    xaxis_title="Total Base Stats",
    yaxis_title="Legendary (T/F)",
)

fig.show()

The values for *legendary* is True/False (0/1) i.e. it is a discrete categorical variable.

When predicted values are discrete, such as *legendary*, then there are a lot of better options than linear regression.

Instead of a line, underlying structure is the sigmoid function

$$ \sigma(t) = \frac{1}{1+e^{-t}} $$

![image](https://miro.medium.com/0*D5do3xhv5ulF50w2.png)

- Often times we write $exp(-t) = e^{-t}
- Bounded between 0 and 1
- It's derivatives make computing loss function and gradient descent straightforward
---


- What is $\sigma(0)$?

$$ \sigma(0) = \frac{1}{1+e^{-0}} = \frac{1}{1+1} = \frac{1}{2}$$

- What is $\sigma(1)$?

$$ \sigma(1) = \frac{1}{1+e^{-1}} = 0.73$$

- What is $\sigma(1000)$?

$$ \sigma(1000) = \frac{1}{1+e^{-1000}} \approx  \frac{1}{1+\text{small}} = 1$$

- What is $\sigma(-1)$?

$$ \sigma(-1) = \frac{1}{1+e^{1}} = 0.27$$

- What is $\sigma(-1000)$?

$$ \sigma(-1000) = \frac{1}{1+e^{1000}} \approx  \frac{1}{\text{big}} = 0$$

---

The logistic model:

$$ f_{\hat\theta}(x) = \sigma(\hat\theta * x)$$

where $\hat\theta$ is a vector of values for each feature of the model.

In [30]:
import numpy as np
import plotly.graph_objects as go

np.random.seed(1)
x = np.random.uniform(-5,5,120)
y = (x + np.random.normal(0,2,120) > 0).astype(int)

def logistic(x, theta):
    return 1/(1+np.exp(-(theta*x)))

def cross_entropy(y, p):
    eps = 1e-9
    return -np.mean(y*np.log(p+eps) + (1-y)*np.log(1-p+eps))

x_curve = np.linspace(-5,5,300)

theta_vals = np.linspace(-4,4,60)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x,
    y=y,
    mode="markers",
    name="data"
))

loss_values = []

for theta in theta_vals:

    p_curve = logistic(x_curve, theta)
    p_data = logistic(x, theta)

    loss = cross_entropy(y, p_data)
    loss_values.append(loss)

    fig.add_trace(go.Scatter(
        x=x_curve,
        y=p_curve,
        mode="lines",
        visible=False,
        name=f"theta={theta:.2f}"
    ))

start = len(theta_vals)//2
fig.data[start+1].visible = True

steps = []
for i,theta in enumerate(theta_vals):

    step = dict(
        method="update",
        args=[
            {"visible":[True]+[j==i for j in range(len(theta_vals))]},
            {"title":f"Logistic Regression — θ = {theta:.2f} | Cross Entropy = {loss_values[i]:.3f}"}
        ],
        label=f"{theta:.2f}"
    )

    steps.append(step)

fig.update_layout(
    sliders=[dict(
        active=start,
        currentvalue={"prefix":"θ: "},
        steps=steps
    )],
    title="Logistic Regression with Cross Entropy",
    xaxis_title="x",
    yaxis_title="Probability",
    template="plotly_white"
)

fig.show()

We will use a loss function that is suited for fitting logistic models.

The cross-entropy loss function

$$ L(\theta, X, y) = \frac{1}{n} \sum_i (-y_i \ln(f_\theta(X_i)) - (1-y_i) \ln(1-f_\theta(X_i)))$$

The key intuitition is:
- The more confident the model is in predicting the correct outcome, the lower the loss.
- The more confident the model is in predicting the wrong outcome, the higher the loss.

The derivative is: $\sigma'(t) = \sigma(t)(1-\sigma(t))$.

Write $\sigma_i = f_{\hat\theta}(X_i \theta)$, then

$$ \nabla_\theta \sigma_i = \sigma_i (1-\sigma_i) X_i$$

We can fit this to a model using gradient descent:

$$\hat\theta = \arg \min_\theta L(\theta*X, y)$$

In [29]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression, LogisticRegression

url = "https://raw.githubusercontent.com/KeithGalli/pandas/master/pokemon_data.csv"
df = pd.read_csv(url)

df["Total"] = df[["HP","Attack","Defense","Sp. Atk","Sp. Def","Speed"]].sum(axis=1)

y = df["Legendary"].astype(int)
X = df[["Total"]].values

lin = LinearRegression().fit(X, y)
log = LogisticRegression().fit(X, y)

x_range = np.linspace(X.min(), X.max(), 300)
x_range = x_range.reshape(-1,1)

lin_pred = lin.predict(x_range)
log_pred = log.predict_proba(x_range)[:,1]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df["Total"],
    y=y,
    mode="markers",
    text=df["Name"],
    name="Pokemon",
    marker=dict(size=8)
))

fig.add_trace(go.Scatter(
    x=x_range.flatten(),
    y=lin_pred,
    mode="lines",
    name="Linear Regression",
    line=dict(width=4)
))

fig.add_trace(go.Scatter(
    x=x_range.flatten(),
    y=log_pred,
    mode="lines",
    name="Logistic Regression",
    line=dict(width=4)
))

fig.update_layout(
    title="Legendary Pokemon: Regression vs Classification",
    xaxis_title="Total Base Stats",
    yaxis_title="Legendary (T/F)",
    template="plotly_white"
)

fig.show()

## Confusion Matrix

In [1]:
import pandas as pd

data = {
    "label": ["spam","spam","spam","ham","ham","ham"],
    "email_text": [
        "Dear E-mail Owner, My name is Jeff Bezos, an American, investor, and charity donor. I'm the founder, CEO and president of Amazon.com,And Your email address has won you ( $2.500,000.00 ) Kindly get back to me , so I know your email address is valid. mrjefferybo600@gmail.com) Best Regards",
        "T-Mobile customer you may now claim your FREE CAMERA PHONE upgrade & a pay & go sim card for your loyalty. Call on 0845 021 3680.Offer ends 28thFeb",
        "U were outbid by simonwatson5120 on the Shinco DVD Plyr. 2 bid again, visit sms. ac/smsrewards 2 end bid notifications, reply END OUT",
        "I know but you need to get hotel now. I just got my invitation but i had to apologise. Cali is to sweet for me to come to some english bloke's weddin",
        "I'm really sorry i won't b able 2 do this friday.hope u can find an alternative.hope yr term's going ok:-)",
        "Lol I know! They're so dramatic. Schools already closed for tomorrow. Apparently we can't drive in the inch of snow were supposed to get"
    ]
}

df = pd.DataFrame(data)

df.style.set_properties(
    subset=["email_text"],
    **{
        "white-space": "pre-wrap",
        "max-width": "500px",
        "font-family": "monospace"
    }
).set_table_styles(
    [{"selector":"th","props":[("text-align","left")]}]
)

,label,email_text
0,spam,"Dear E-mail Owner, My name is Jeff Bezos, an American, investor, and charity donor. I'm the founder, CEO and president of Amazon.com,And Your email address has won you ( $2.500,000.00 ) Kindly get back to me , so I know your email address is valid. mrjefferybo600@gmail.com) Best Regards"
1,spam,T-Mobile customer you may now claim your FREE CAMERA PHONE upgrade & a pay & go sim card for your loyalty. Call on 0845 021 3680.Offer ends 28thFeb
2,spam,"U were outbid by simonwatson5120 on the Shinco DVD Plyr. 2 bid again, visit sms. ac/smsrewards 2 end bid notifications, reply END OUT"
3,ham,I know but you need to get hotel now. I just got my invitation but i had to apologise. Cali is to sweet for me to come to some english bloke's weddin
4,ham,I'm really sorry i won't b able 2 do this friday.hope u can find an alternative.hope yr term's going ok:-)
5,ham,Lol I know! They're so dramatic. Schools already closed for tomorrow. Apparently we can't drive in the inch of snow were supposed to get


- Our goal is to build a classifier that can identiy spam email vs non-spam (aka ham)
- There are may approaches to this classic problem
- We want to measure solutions on how often the model predicts correctly and incorrectly

Standard metrics measure how often our models predicts correctly and incorrectly

Many are combination of 4 basic measures:
- True Positive: corectly labeled with positive class
- False Negative: belongs to a positive class but mislabeled as negative
- False Positive: belongs to a negative class but mislabeled as positive
- True Negative: correctly labeled with negative class

Often organized in a confusion matrix: a heatmap of model predictions vs actual labels (we will go more into depth with this later)

In [56]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.linear_model import LogisticRegression

url = "https://raw.githubusercontent.com/KeithGalli/pandas/master/pokemon_data.csv"
df = pd.read_csv(url)

df["Total"] = df[["HP","Attack","Defense","Sp. Atk","Sp. Def","Speed"]].sum(axis=1)

X = df["Total"].values
y = df["Legendary"].astype(int).values

log_model = LogisticRegression()
log_model.fit(X.reshape(-1,1), y)
x_range = np.linspace(X.min(), X.max(), 300).reshape(-1,1)
log_pred = log_model.predict_proba(x_range)[:,1]

theta_vals = np.linspace(X.min(), X.max(), 50)
frames = []

for theta in theta_vals:
    preds = (X >= theta).astype(int)
    
    TN = np.sum((preds==0) & (y==0))
    FP = np.sum((preds==1) & (y==0))
    FN = np.sum((preds==0) & (y==1))
    TP = np.sum((preds==1) & (y==1))
    

    z_matrix = [[FP, TP],
                [TN, FN]] 
    
    scatter_left = [
        go.Scatter(
            x=X,
            y=y,
            mode="markers",
            text=df["Name"],
            marker=dict(
                color=["red" if p==1 else "blue" for p in preds],
                size=8
            ),
            name="Pokemon",
            xaxis="x1",
            yaxis="y1"
        ),
        go.Scatter(
            x=[theta, theta],
            y=[-0.05, 1.05],
            mode="lines",
            line=dict(color="black", width=3, dash="dash"),
            name="Theta",
            xaxis="x1",
            yaxis="y1"
        ),
        go.Scatter(
            x=x_range.flatten(),
            y=log_pred,
            mode="lines",
            line=dict(width=4, color="green"),
            name="Logistic Regression",
            xaxis="x1",
            yaxis="y1"
        )
    ]
    
    heatmap_right = [
        go.Heatmap(
            z=z_matrix,
            x=["Pred 0","Pred 1"],
            y=["Actual 1","Actual 0"],
            text=[[f"FP={FP}", f"TP={TP}"], [f"TN={TN}", f"FN={FN}"]],
            texttemplate="%{text}",
            colorscale="Blues",
            showscale=False,
            xaxis="x2",
            yaxis="y2"
        )
    ]
    
    frames.append(go.Frame(data=scatter_left + heatmap_right, name=str(theta)))

init_idx = len(theta_vals)//2
theta0 = theta_vals[init_idx]
preds0 = (X >= theta0).astype(int)

TN = np.sum((preds0==0) & (y==0))
FP = np.sum((preds0==1) & (y==0))
FN = np.sum((preds0==0) & (y==1))
TP = np.sum((preds0==1) & (y==1))
z_matrix0 = [[FP, TP],[TN, FN]]

fig = go.Figure(
    data=[
        go.Scatter(
            x=X,
            y=y,
            mode="markers",
            text=df["Name"],
            marker=dict(
                color=["red" if p==1 else "blue" for p in preds0],
                size=8
            ),
            name="Pokemon",
            xaxis="x1",
            yaxis="y1"
        ),
        go.Scatter(
            x=[theta0, theta0],
            y=[-0.05,1.05],
            mode="lines",
            line=dict(color="black", width=3, dash="dash"),
            name="Theta",
            xaxis="x1",
            yaxis="y1"
        ),
        go.Scatter(
            x=x_range.flatten(),
            y=log_pred,
            mode="lines",
            line=dict(width=4, color="green"),
            name="Logistic Regression",
            xaxis="x1",
            yaxis="y1"
        ),
        go.Heatmap(
            z=z_matrix0,
            x=["Pred 0","Pred 1"],
            y=["Actual 1","Actual 0"],
            text=[[f"FP={FP}", f"TP={TP}"], [f"TN={TN}", f"FN={FN}"]],
            texttemplate="%{text}",
            colorscale="Blues",
            showscale=False,
            xaxis="x2",
            yaxis="y2"
        )
    ],
    frames=frames
)

fig.update_layout(
    title="Pokemon Classification Using Threshold Theta",
    template="plotly_white",
    xaxis=dict(domain=[0,0.45], title="Total Stats", anchor="y1"),
    yaxis=dict(domain=[0,1], title="Legendary (0/1)", anchor="x1"),
    xaxis2=dict(domain=[0.55,1], title="Predicted", anchor="y2"),
    yaxis2=dict(domain=[0,1], title="Actual", anchor="x2"),
    sliders=[dict(
        active=init_idx,
        currentvalue={"prefix":"Theta θ: "},
        pad={"t":50},
        steps=[dict(
            method="animate",
            args=[[str(t)], {"mode":"immediate","frame":{"duration":0,"redraw":True}}],
            label=f"{t:.0f}"
        ) for t in theta_vals]
    )]
)

fig.show()

## Linear Algebra Recap

### Vectors

A **vector** of length $ n $ is just a sequence (or array, or tuple) of $ n $ numbers, which we write as $ \hat x = (x_1, \ldots, x_n) $ or  $ x = [x_1, \ldots, x_n] $.

The set of all $ n $-vectors is denoted by $ \mathbb R^n $.

For example, $ \mathbb R ^2 $ is the plane, and a vector in $ \mathbb R^2 $ is just a point in the plane.

Traditionally, vectors are represented visually as arrows from the origin to
the point.


In [ ]:
import plotly.graph_objects as go
import numpy as np

In [ ]:
vecs = [(2, 4), (-3, 3), (-4, -3.5)]

fig = go.Figure()

for v in vecs:
    fig.add_annotation(
        x=v[0], y=v[1],
        ax=0, ay=0,
        xref="x", yref="y",
        axref="x", ayref="y",
        showarrow=True,
        arrowhead=3,
        arrowsize=1.5,
        arrowwidth=3,
        arrowcolor="blue"
    )
    fig.add_annotation(
        x=1.1 * v[0],
        y=1.1 * v[1],
        text=str(v),
        showarrow=False
    )

fig.update_xaxes(
    zeroline=True, zerolinewidth=2, zerolinecolor='black',
    range=[-5, 5]
)
fig.update_yaxes(
    zeroline=True, zerolinewidth=2, zerolinecolor='black',
    range=[-5, 5]
)

fig.update_layout(
    width=800,
    height=800,
    template='plotly_white',
    xaxis=dict(showgrid=True, gridwidth=1, gridcolor='lightgray'),
    yaxis=dict(showgrid=True, gridwidth=1, gridcolor='lightgray')
)

fig.show()

The two most common operators for vectors are addition and scalar multiplication.

When we add two vectors, we add them element-by-element

$$
x + y =
\begin{bmatrix}
    x_1 \\
    x_2 \\
    \vdots \\
    x_n
\end{bmatrix} +
\begin{bmatrix}
     y_1 \\
     y_2 \\
    \vdots \\
     y_n
\end{bmatrix} :=
\begin{bmatrix}
    x_1 + y_1 \\
    x_2 + y_2 \\
    \vdots \\
    x_n + y_n
\end{bmatrix}
$$

Scalar multiplication is an operation that takes a number $ \gamma $ and a vector $ x $ and produces

$$
\gamma x :=
\begin{bmatrix}
    \gamma x_1 \\
    \gamma x_2 \\
    \vdots \\
    \gamma x_n
\end{bmatrix}
$$




In [ ]:
x = np.array([2, 2])
scalars = [-2, 2]

fig = go.Figure()

fig.add_annotation(
    x=x[0], y=x[1],
    ax=0, ay=0,
    xref="x", yref="y",
    axref="x", ayref="y",
    showarrow=True,
    arrowhead=3,
    arrowsize=1.5,
    arrowwidth=3,
    arrowcolor="blue"
)
fig.add_annotation(
    x=x[0] + 0.4,
    y=x[1] - 0.2,
    text='x',
    showarrow=False,
    font=dict(size=16)
)

for s in scalars:
    v = s * x
    fig.add_annotation(
        x=v[0], y=v[1],
        ax=0, ay=0,
        xref="x", yref="y",
        axref="x", ayref="y",
        showarrow=True,
        arrowhead=3,
        arrowsize=1.5,
        arrowwidth=3,
        arrowcolor="red",
        opacity=0.5
    )
    fig.add_annotation(
        x=v[0] + 0.4,
        y=v[1] - 0.2,
        text=f'{s}x',
        showarrow=False,
        font=dict(size=16)
    )

fig.update_xaxes(
    zeroline=True, zerolinewidth=2, zerolinecolor='black',
    range=[-5, 5]
)
fig.update_yaxes(
    zeroline=True, zerolinewidth=2, zerolinecolor='black',
    range=[-5, 5]
)

fig.update_layout(
    width=800,
    height=800,
    template='plotly_white',
    xaxis=dict(showgrid=True, gridwidth=1, gridcolor='lightgray'),
    yaxis=dict(showgrid=True, gridwidth=1, gridcolor='lightgray')
)

fig.show()

In Python, a vector can be represented as a list or tuple, such as `x = (2, 4, 6)`, but is more commonly
represented as a [NumPy array](https://python-programming.quantecon.org/numpy.html#numpy-arrays).

One advantage of NumPy arrays is that scalar multiplication and addition have very natural syntax

In [ ]:
x = np.ones(3)            # Vector of three ones
y = np.array((2, 4, 6))   # Converts tuple (2, 4, 6) into array
x + y

array([3., 5., 7.])

### Inner Products

The **inner product**(or dot product) of vectors $ x,y \in \mathbb R ^n $ is defined as

$$
x' y := \sum_{i=1}^n x_i y_i
$$

Two vectors are called **orthogonal** if their inner product is zero.

The **norm** of a vector $ x $ represents its "length" (i.e., its distance from the zero vector) and is defined as

$$
\| x \| := \sqrt{x' x} := \left( \sum_{i=1}^n x_i^2 \right)^{1/2}
$$

The expression $ \| x - y\| $ is thought of as the distance between $ x $ and $ y $.

It can be computed as follows:


In [ ]:
np.sum(x * y)          # Inner product of x and y, method 1

np.float64(12.0)

In [ ]:
x @ y                  # Inner product of x and y, method 2 (preferred)

np.float64(12.0)

The `@` operator is preferred because it uses optimized BLAS libraries that implement fused multiply-add operations, providing better performance and numerical accuracy compared to the separate multiply and sum operations.

In [ ]:
np.sqrt(np.sum(x**2))  # Norm of x, take one

np.float64(1.7320508075688772)

In [ ]:
np.sqrt(x @ x)         # Norm of x, take two (preferred)

np.float64(1.7320508075688772)

In [ ]:
np.linalg.norm(x)      # Norm of x, take three

np.float64(1.7320508075688772)

![image](https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fdanjcalderone.github.io%2Fdcmath%2Ffigs%2Fstills%2FVEC%2FINNERPROD_DIAG.png&f=1&nofb=1&ipt=ac41c081b89c544ee3243496b62121fa988b8b6ac517f518ed8aef4ca935af23)

In [ ]:
thetas = np.linspace(0, 2*np.pi, 60)
v = np.array([1, 0])

frames = []

for theta in thetas:
    w = np.array([np.cos(theta), np.sin(theta)])

    dot = np.dot(v, w)

    proj = dot * v

    data = [
        go.Scatter(x=[0, v[0]], y=[0, v[1]],
                   mode='lines+markers',
                   name='v = first vector',
                   line=dict(width=4)),

        go.Scatter(x=[0, w[0]], y=[0, w[1]],
                   mode='lines+markers',
                   name='w = second vector',
                   line=dict(width=4)),

        go.Scatter(x=[0, proj[0]], y=[0, proj[1]],
                   mode='lines+markers',
                   name='projection',
                   line=dict(dash='dash', width=4)),

        go.Scatter(x=[w[0], proj[0]], y=[w[1], proj[1]],
                   mode='lines',
                   showlegend=False,
                   line=dict(dash='dot'))
    ]

    frames.append(go.Frame(
        data=data,
        name=str(round(theta,2)),
        layout=go.Layout(
            title=f"Dot Product = {dot:.2f}"
        )
    ))

fig = go.Figure(
    data=frames[0].data,
    frames=frames
)

fig.update_layout(
    sliders=[{
        "steps": [
            {"args": [[f.name],
                      {"frame": {"duration": 50, "redraw": True},
                       "mode": "immediate"}],
             "label": f.name,
             "method": "animate"}
            for f in frames
        ],
        "currentvalue": {"prefix": "θ: "}
    }],
    xaxis=dict(range=[-1.5,1.5]),
    yaxis=dict(range=[-1.5,1.5]),
    width=700,
    height=700
)

fig.show()

### Eigenvectors and eigenvalues
Given a square $n \times n$ matrix, a scalar $\lambda$ is called an eigenvalue of $A$ if there exists some nonzero vector $V$ in $\mathbb{R}^n$ such that $AV=\lambda V$. The vector $V$ is the eigenvector associated with $\lambda$. The equation states that when an eigenvalue of $A$ is multiplied with $A$, the result is simply a multiple of the eigenvector.

$$
\begin{bmatrix}
        1 & 2 \\
        2 & 1
    \end{bmatrix}
    \begin{bmatrix}
        1 \\
        3
    \end{bmatrix}
    =
    \begin{bmatrix}
        7 \\
        5
    \end{bmatrix}
$$

Here, the matrix 
$
A = \begin{bmatrix} 1 & 2 \\ 2 & 1 \end{bmatrix}
$
transforms the vector 
$
x = \begin{bmatrix} 1 \\ 3 \end{bmatrix}
$
to the vector 
$
y = \begin{bmatrix} 7 \\ 5 \end{bmatrix}$.

In [ ]:
import numpy as np
import plotly.graph_objects as go

v = np.array([1, 3])
Av = np.array([7, 5])

proj = (np.dot(v, Av) / np.dot(Av, Av)) * Av

fig = go.Figure()
fig.add_shape(type="line", x0=-2, y0=0, x1=8, y1=0,
              line=dict(width=2))
fig.add_shape(type="line", x0=0, y0=-2, x1=0, y1=6,
              line=dict(width=2))

fig.add_annotation(
    x=v[0], y=v[1],
    ax=0, ay=0,
    xref="x", yref="y",
    axref="x", ayref="y",
    showarrow=True,
    arrowhead=3,
    arrowwidth=2
)

fig.add_annotation(
    x=Av[0], y=Av[1],
    ax=0, ay=0,
    xref="x", yref="y",
    axref="x", ayref="y",
    showarrow=True,
    arrowhead=3,
    arrowwidth=2
)

fig.add_annotation(
    x=proj[0], y=proj[1],
    ax=0, ay=0,
    xref="x", yref="y",
    axref="x", ayref="y",
    showarrow=True,
    arrowhead=3,
    arrowwidth=2
)

fig.add_annotation(x=1.2, y=3.2, text="x = (1,3)", showarrow=False)
fig.add_annotation(x=7.2, y=5.2, text="Ax = (7,5)", showarrow=False)

fig.update_layout(
    xaxis=dict(range=[-2, 8], zeroline=False),
    yaxis=dict(range=[-2, 6], zeroline=False),
    width=700,
    height=500,
    title="Vector Transformation under Matrix A"
)

fig.update_yaxes(scaleanchor="x", scaleratio=1)

fig.show()

Thus, an eigenvector of $ A $ is a nonzero vector $ v $ such that when the map $ A $ is
applied, $ v $ is merely scaled.

In [ ]:
from numpy.linalg import eig

A = np.array([[1, 2],
              [2, 1]])

evals, evecs = eig(A)

print(*[f"λ = {evals[i]:.2f}, v = {evecs[:,i]}" for i in range(len(evals))], sep="\n")

λ = 3.00, v = [0.70710678 0.70710678]
λ = -1.00, v = [-0.70710678  0.70710678]


In [ ]:
A = np.array([[1, 2],
              [2, 1]])

evals, evecs = eig(A)
evecs = [evecs[:, 0], evecs[:, 1]]

xmin, xmax = -3, 3
ymin, ymax = -3, 3

fig = go.Figure()

fig.add_shape(type="line", x0=xmin, y0=0, x1=xmax, y1=0,
              line=dict(width=2))
fig.add_shape(type="line", x0=0, y0=ymin, x1=0, y1=ymax,
              line=dict(width=2))

for v in evecs:
    fig.add_annotation(
        x=v[0], y=v[1],
        ax=0, ay=0,
        xref="x", yref="y",
        axref="x", ayref="y",
        showarrow=True,
        arrowhead=3,
        arrowsize=1,
        arrowwidth=2,
        opacity=0.7
    )

for v in evecs:
    Av = A @ v
    fig.add_annotation(
        x=Av[0], y=Av[1],
        ax=0, ay=0,
        xref="x", yref="y",
        axref="x", ayref="y",
        showarrow=True,
        arrowhead=3,
        arrowsize=1,
        arrowwidth=2,
        opacity=0.7
    )

x_vals = np.linspace(xmin, xmax, 100)

for v in evecs:
    slope = v[1] / v[0]
    fig.add_trace(go.Scatter(
        x=x_vals,
        y=slope * x_vals,
        mode='lines',
        line=dict(width=1),
        showlegend=False
    ))

fig.update_layout(
    title="Eigenvectors ",
    xaxis=dict(range=[xmin, xmax], zeroline=False),
    yaxis=dict(range=[ymin, ymax], zeroline=False),
    width=700,
    height=700
)

fig.update_yaxes(scaleanchor="x", scaleratio=1)

fig.show()

The eigenvalue equation is equivalent to $ (A - \lambda I) v = 0 $.

This equation has a nonzero solution $ v $ only when the columns of $ A - \lambda I $ are linearly dependent.

This in turn is equivalent to stating the determinant is zero.

Hence, to find all eigenvalues, we can look for $ \lambda $ such that the
determinant of $ A - \lambda I $ is zero.

This problem can be expressed as one of solving for the roots of a polynomial
in $ \lambda $ of degree $ n $.

This in turn implies the existence of $ n $ solutions in the complex
plane, although some might be repeated.

## Support Vector Machines (SVM)

Support vector machines (SVMs) are a particularly powerful and flexible class of supervised algorithms for both classification and regression. In this section, we will develop the intuition behind support vector machines and their use in classification problems.

We simply find a line or curve (in two dimensions) or manifold (in multiple dimensions) that divides the classes from each other.

As an example of this, consider the simple case of a classification task, in which the two classes of points are well separated:

In [ ]:
from sklearn.datasets import make_blobs
X, y = make_blobs(n_samples=50, centers=2,
                  random_state=0, cluster_std=0.60)

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=X[:, 0],
    y=X[:, 1],
    mode='markers',
    marker=dict(
        color=y,
        colorscale='YlOrRd',
        size=10,
        line=dict(width=1, color='black')
    ),
    name='Blobs'
))

fig.update_layout(
    title='2D Blob Scatter',
    xaxis_title='X1',
    yaxis_title='X2',
    width=700,
    height=600
)

fig.show()

A linear discriminative classifier would attempt to draw a straight line separating the two sets of data, and thereby create a model for classification. For two dimensional data like that shown here, this is a task we could do by hand. But immediately we see a problem: there is more than one possible dividing line that can perfectly discriminate between the two classes!

We can draw them as follows:

In [ ]:
fig = go.Figure()

xfit = np.linspace(-1, 3.5, 100)
lines = [(1, 0.65), (0.5, 1.6), (-0.2, 2.9)]

fig.add_trace(go.Scatter(
    x=X[:, 0],
    y=X[:, 1],
    mode='markers',
    marker=dict(
        color=y,
        colorscale='YlOrRd',
        size=10,
        line=dict(width=1, color='black')
    ),
    name='Data points'
))

fig.add_trace(go.Scatter(
    x=[0.6],
    y=[2.1],
    mode='markers',
    marker=dict(color='red', symbol='x', size=12, line=dict(width=2)),
    name='Special point'
))

for m, b in lines:
    fig.add_trace(go.Scatter(
        x=xfit,
        y=m * xfit + b,
        mode='lines',
        line=dict(color='black'),
        showlegend=False
    ))

fig.update_layout(
    title='2D Blob Seperation',
    xaxis=dict(range=[-1, 3.5]),
    yaxis=dict(range=[min(X[:,1])-0.5, max(X[:,1])+0.5]),
    width=700,
    height=500
)

fig.show()

These are three very different separators which, nevertheless, perfectly discriminate between these samples. Depending on which you choose, a new data point (e.g., the one marked by the "X" in this plot) will be assigned a different label! Evidently our simple intuition of "drawing a line between classes" is not enough, and we need to think a bit deeper.

Support vector machines offer one way to improve on this. The intuition is this: rather than simply drawing a zero-width line between the classes, we can draw around each line a margin of some width, up to the nearest point. Here is an example of how this might look:

In [ ]:
xfit = np.linspace(-1, 3.5, 200)
lines = [(1, 0.65, 0.33), (0.5, 1.6, 0.55), (-0.2, 2.9, 0.2)]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=X[:, 0],
    y=X[:, 1],
    mode='markers',
    marker=dict(
        color=y,
        colorscale='YlOrRd',
        size=10,
        line=dict(width=1, color='black')
    ),
    name='Data points'
))

for m, b, d in lines:
    yfit = m * xfit + b
    y_lower = yfit - d
    y_upper = yfit + d
    
    fig.add_trace(go.Scatter(
        x=np.concatenate([xfit, xfit[::-1]]),
        y=np.concatenate([y_upper, y_lower[::-1]]),
        fill='toself',
        fillcolor='rgba(170,170,170,0.4)',
        line=dict(color='rgba(255,255,255,0)'), 
        showlegend=False,
        name='Band'
    ))
    
    fig.add_trace(go.Scatter(
        x=xfit,
        y=yfit,
        mode='lines',
        line=dict(color='black'),
        showlegend=False
    ))

fig.update_layout(
    title='Boundaries with Bands',
    xaxis=dict(range=[-1, 3.5]),
    yaxis=dict(range=[min(X[:,1])-0.5, max(X[:,1])+0.5]),
    width=700,
    height=500
)

fig.show()

In support vector machines, the line that maximizes this margin is the one we will choose as the optimal model. Support vector machines are an example of such a maximum margin estimator.

### Fitting a SVM

Let's see the result of an actual fit to this data: we will use Scikit-Learn's support vector classifier to train an SVM model on this data. For the time being, we will use a linear kernel and set the C parameter to a very large number (we'll discuss the meaning of these in more depth momentarily).

In [ ]:
from sklearn.svm import SVC # "Support vector classifier"
model = SVC(kernel='linear', C=1E10)
model.fit(X, y)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",10000000000.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'linear'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",False
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


To better visualize what's happening here, let's create a quick convenience function that will plot SVM decision boundaries for us:

In [ ]:
def plot_svc_decision_function_plotly(model, X, y, plot_support=True, grid_points=200):
    x_min, x_max = X[:,0].min() - 1, X[:,0].max() + 1
    y_min, y_max = X[:,1].min() - 1, X[:,1].max() + 1
    xx = np.linspace(x_min, x_max, grid_points)
    yy = np.linspace(y_min, y_max, grid_points)
    YY, XX = np.meshgrid(yy, xx)
    grid = np.c_[XX.ravel(), YY.ravel()]
    
    Z = model.decision_function(grid).reshape(XX.shape).T

    fig = go.Figure()

    # Decision boundary
    fig.add_trace(go.Contour(
        x=xx,
        y=yy,
        z=Z,
        showscale=False,
        contours=dict(start=0, end=0, coloring='lines', showlabels=True,
                      labelfont=dict(size=12, color='black')),
        line=dict(color='black', width=2),
        hoverinfo='skip'
    ))

    # Margins
    for level in [-1, 1]:
        fig.add_trace(go.Contour(
            x=xx,
            y=yy,
            z=Z,
            showscale=False,
            contours=dict(start=level, end=level, coloring='lines', showlabels=True,
                          labelfont=dict(size=12, color='black')),
            line=dict(color='black', width=2, dash='dash'),
            hoverinfo='skip'
        ))

    # Data points
    fig.add_trace(go.Scatter(
        x=X[:,0], y=X[:,1],
        mode='markers',
        marker=dict(color=y, colorscale='Viridis', size=10,
                    line=dict(width=1, color='black')),
        name='Data points'
    ))

    # Support vectors
    if plot_support:
        fig.add_trace(go.Scatter(
            x=model.support_vectors_[:,0], y=model.support_vectors_[:,1],
            mode='markers',
            marker=dict(size=15, color='rgba(0,0,0,0)',
                        line=dict(color='red', width=2)),
            name='Support vectors'
        ))

    fig.update_layout(
        title='SVC Decision Function (Plotly)',
        xaxis_title='X1',
        yaxis_title='X2',
        width=700,
        height=600
    )

    return fig

In [ ]:
model = SVC(kernel='linear', C=1e5)
model.fit(X, y)

fig = plot_svc_decision_function_plotly(model, X, y)
fig.show()

This is the dividing line that maximizes the margin between the two sets of points. Notice that a few of the training points just touch the margin: they are indicated by the black circles in this figure. These points are the pivotal elements of this fit, and are known as the support vectors, and give the algorithm its name. In Scikit-Learn, the identity of these points are stored in the support_vectors_ attribute of the classifier:

In [ ]:
model.support_vectors_

array([[0.44359863, 3.11530945],
       [2.33812285, 3.43116792],
       [2.06156753, 1.96918596]])

A key to this classifier's success is that for the fit, only the position of the support vectors matter; any points further from the margin which are on the correct side do not modify the fit! Technically, this is because these points do not contribute to the loss function used to fit the model, so their position and number do not matter so long as they do not cross the margin.

Next time: Kernels!

## MNIST Classification

MNIST or Modified National Institute of Standards & Technology, is a dataset consisting of 1797 scans of handwritten digits.

Each entry has the digit represented as well as the 64 values representing the grey scale for a 8x8 image, for example:

In [69]:
import numpy as np
from sklearn.datasets import fetch_openml
import plotly.graph_objects as go

mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X, y = mnist['data'], mnist['target'].astype(int)
X = X / 255.0

num_images = 10
images = X[:num_images].reshape(-1,28,28)
labels = y[:num_images]

init_idx = 0

fig = go.Figure(
    data=[
        go.Heatmap(
            z=images[init_idx][::-1], 
            colorscale='gray',
            showscale=False
        )
    ]
)

steps = []
for i in range(num_images):
    step = dict(
        method='update',
        args=[{'z':[images[i][::-1]]}, 
              {'title': f"MNIST Image Index {i} (Label: {labels[i]})"}],
        label=str(i)
    )
    steps.append(step)

sliders = [dict(
    active=0,
    currentvalue={"prefix":"Image Index: "},
    pad={"t":50},
    steps=steps
)]

fig.update_layout(
    sliders=sliders,
    title=f"MNIST Image Index {init_idx} (Label: {labels[init_idx]})",
    xaxis=dict(showticklabels=False),
    yaxis=dict(showticklabels=False)
)

fig.show()

General Strategy:
- Clean the data
- Split the data into training and testing subsets
- Instantiate and fit the model to the training data (validate/tune model parameters)
- Test the model

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X, y = mnist['data'], mnist['target'].astype(int)
X = X / 255.0

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

log_clf = LogisticRegression(max_iter=100)
log_clf.fit(X_train, y_train)

y_pred = log_clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.9202857142857143


/Users/ko/Documents/data-science-sp26/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning:

lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression



In [ ]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import plotly.graph_objects as go
from plotly.subplots import make_subplots

mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X, y = mnist['data'], mnist['target'].astype(int)
X = X / 255.0

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

log_clf = LogisticRegression(max_iter=100)
log_clf.fit(X_train, y_train)

init_idx = 0
digit_image = X_test[init_idx].reshape(28,28)
probs = log_clf.predict_proba(X_test[init_idx].reshape(1,-1))[0]

fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.4,0.6],
    specs=[[{"type":"heatmap"}, {"type":"bar"}]],
    subplot_titles=["MNIST Digit", "Predicted Probabilities"]
)

fig.add_trace(
    go.Heatmap(
        z=digit_image[::-1],
        colorscale="gray",
        showscale=False
    ),
    row=1, col=1
)

fig.add_trace(
    go.Bar(
        x=list(range(10)),
        y=probs,
        text=[f"{p:.2f}" for p in probs],
        textposition="auto",
        name="Probabilities"
    ),
    row=1, col=2
)

steps = []
for i in range(100):
    digit_i = X_test[i].reshape(28,28)
    probs_i = log_clf.predict_proba(X_test[i].reshape(1,-1))[0]
    step = dict(
        method="update",
        args=[
            {"z":[digit_i[::-1], None], "y":[None, probs_i], "text":[None, [f"{p:.2f}" for p in probs_i]]}
        ],
        label=str(i)
    )
    steps.append(step)

sliders = [dict(active=init_idx, currentvalue={"prefix":"Test index: "}, pad={"t":50}, steps=steps)]

fig.update_layout(
    sliders=sliders,
    title="MNIST Logistic Regression Prediction Demo",
    xaxis=dict(showticklabels=False),
    yaxis=dict(showticklabels=False)
)

fig.show()

/Users/ko/Documents/data-science-sp26/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning:

lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression

